In [0]:
%sql
-- Create a table with outlier flags for further investigation

CREATE OR REPLACE TABLE oulad.clean.student_vle_with_flags AS
SELECT 
  *,
  -- Flag outliers and edge cases
  CASE 
    WHEN sum_click > 1000 THEN 'high_click_outlier'
    WHEN sum_click > 5000 THEN 'extreme_click_outlier'
    ELSE 'normal'
  END as click_outlier_flag,
  
  CASE 
    WHEN date < -20 THEN 'pre_course_access'
    WHEN date > 250 THEN 'late_course_access'
    WHEN date < 0 THEN 'pre_start_normal'
    ELSE 'normal'
  END as date_outlier_flag,
  
  -- Add a composite quality flag
  CASE 
    WHEN sum_click > 1000 OR date < -20 OR date > 250 THEN 'review_needed'
    ELSE 'clean'
  END as quality_flag
FROM oulad.clean.student_vle;

-- Get counts of flagged records by type
SELECT 
  quality_flag,
  click_outlier_flag,
  date_outlier_flag,
  COUNT(*) as record_count,
  COUNT(DISTINCT id_student) as student_count,
  MIN(sum_click) as min_clicks,
  MAX(sum_click) as max_clicks,
  AVG(sum_click) as avg_clicks,
  MIN(date) as min_date,
  MAX(date) as max_date
FROM oulad.clean.student_vle_with_flags
GROUP BY quality_flag, click_outlier_flag, date_outlier_flag
ORDER BY record_count DESC;